<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.1-multi-provider-api/notebooks/exercises-5_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.1: Multi-Provider API Integration

*Module 5 · Production Applications with GenAI APIs*

> Gemini goes down for 2 hours. OpenAI hits rate limits. Anthropic changes pricing. If your app depends on a single provider, one outage kills your product. This lesson teaches you to build bulletproof AI systems that work across ALL providers.

---

## About this notebook

This notebook contains the **8 runnable code examples** from the published lesson `lesson-5.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Three SDKs Side by Side — Same Task, Three APIs — `part1a_gemini.py`
2. Step 1 — Three SDKs Side by Side — Same Task, Three APIs — `part1b_openai.py`
3. Step 1 — Three SDKs Side by Side — Same Task, Three APIs — `part1c_anthropic.py`
4. Step 2 — Unified Wrapper — One Interface for All Providers — `part2_unified_wrapper.py`
5. Step 3 — Fallback Routing — Automatic Failover — `part3_fallback_router.py`
6. Step 4 — Smart Routing — Best Model for Each Task — `part4_smart_router.py`
7. Step 5 — Cost Tracking — Know Where Every Dollar Goes — `part5_cost_tracker.py`
8. Step 6 — Project: Production Multi-Provider System — `project_production_system.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q anthropic>=0.40.0 google-generativeai openai transformers requests


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export ANTHROPIC_API_KEY=sk-...
#   export GOOGLE_API_KEY=sk-...
#   export OPENAI_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("ANTHROPIC_API_KEY", "YOUR_ANTHROPIC_API_KEY_HERE")
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")
os.environ.setdefault("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Three SDKs Side by Side — Same Task, Three APIs

See how the same question is asked to Gemini, OpenAI, and Anthropic.

**`part1a_gemini.py`** · _Block 1 of 8_

PROVIDER 1: Google Gemini — pip install google-generativeai


In [ ]:
# =============================================
# PROVIDER 1: Google Gemini
# pip install google-generativeai
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

response = model.generate_content("Explain RAG in 2 sentences.")
print(f"Gemini: {response.text}")

# With system prompt
model = genai.GenerativeModel("gemini-2.0-flash",
    system_instruction="You are a helpful AI tutor.")
chat = model.start_chat()
response = chat.send_message("What is RAG?")
print(f"Gemini (chat): {response.text}")


**`part1b_openai.py`** · _Block 2 of 8_

PROVIDER 2: OpenAI (GPT-4o) — pip install openai


In [ ]:
# =============================================
# PROVIDER 2: OpenAI (GPT-4o)
# pip install openai
# =============================================

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful AI tutor."},
        {"role": "user", "content": "Explain RAG in 2 sentences."},
    ],
    temperature=0.3,
)

print(f"OpenAI: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")


**`part1c_anthropic.py`** · _Block 3 of 8_

PROVIDER 3: Anthropic (Claude) — pip install anthropic


In [ ]:
# =============================================
# PROVIDER 3: Anthropic (Claude)
# pip install anthropic
# =============================================

from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a helpful AI tutor.",
    messages=[
        {"role": "user", "content": "Explain RAG in 2 sentences."},
    ],
)

print(f"Claude: {response.content[0].text}")
print(f"Tokens: {response.usage.input_tokens + response.usage.output_tokens}")


> **What just happened?** Same question ("Explain RAG in 2 sentences"), three different SDKs: Gemini uses generate_content() , OpenAI uses chat.completions.create() , Anthropic uses messages.create() . Each has slightly different parameter names (system_instruction vs system vs system), response structures (response.text vs response.choices[0].message.content vs response.content[0].text), and token counting. The differences are cosmetic. The concept is identical. Now let's unify them.


### Step 2 · Unified Wrapper — One Interface for All Providers

Write code ONCE. Swap providers by changing one string. No code changes needed.

**`part2_unified_wrapper.py`** · _Block 4 of 8_

UNIFIED LLM WRAPPER — One interface. Three providers.


In [ ]:
# =============================================
# UNIFIED LLM WRAPPER
# One interface. Three providers.
# Swap providers by changing one string.
# =============================================

import os, time
from dataclasses import dataclass

@dataclass
class LLMResponse:
    """Unified response format — same regardless of provider."""
    text: str
    provider: str
    model: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    cost_usd: float

class UnifiedLLM:
    """Call any LLM provider through one consistent interface."""
    
    # Pricing per 1M tokens (approximate, 2025-2026)
    PRICING = {
        "gemini-2.0-flash":          {"input": 0.075, "output": 0.30},
        "gemini-2.5-pro":            {"input": 1.25,  "output": 10.00},
        "gpt-4o":                    {"input": 2.50,  "output": 10.00},
        "gpt-4o-mini":               {"input": 0.15,  "output": 0.60},
        "claude-sonnet-4-20250514":  {"input": 3.00,  "output": 15.00},
        "claude-haiku-3-5":          {"input": 0.80,  "output": 4.00},
    }
    
    def __init__(self):
        self._clients = {}
        self._init_providers()
    
    def _init_providers(self):
        # Initialize available providers (only if API key exists)
        if os.getenv("GOOGLE_API_KEY"):
            import google.generativeai as genai
            genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
            self._clients["gemini"] = genai
        
        if os.getenv("OPENAI_API_KEY"):
            from openai import OpenAI
            self._clients["openai"] = OpenAI()
        
        if os.getenv("ANTHROPIC_API_KEY"):
            from anthropic import Anthropic
            self._clients["anthropic"] = Anthropic()
        
        print(f"Initialized providers: {list(self._clients.keys())}")
    
    def _calc_cost(self, model, input_tok, output_tok):
        pricing = self.PRICING.get(model, {"input": 0, "output": 0})
        return (input_tok * pricing["input"] + output_tok * pricing["output"]) / 1_000_000
    
    def _detect_provider(self, model):
        if "gemini" in model: return "gemini"
        if "gpt" in model: return "openai"
        if "claude" in model: return "anthropic"
        raise ValueError(f"Unknown model: {model}")
    
    def chat(self, message: str, model: str = "gemini-2.0-flash",
            system: str = "", temperature: float = 0.3) -> LLMResponse:
        """Send a message to any provider. Returns unified response."""
        provider = self._detect_provider(model)
        start = time.time()
        
        if provider == "gemini":
            m = self._clients["gemini"].GenerativeModel(model,
                system_instruction=system or None,
                generation_config={"temperature": temperature})
            resp = m.generate_content(message)
            text = resp.text
            in_tok = resp.usage_metadata.prompt_token_count
            out_tok = resp.usage_metadata.candidates_token_count
        
        elif provider == "openai":
            msgs = []
            if system: msgs.append({"role": "system", "content": system})
            msgs.append({"role": "user", "content": message})
            resp = self._clients["openai"].chat.completions.create(
                model=model, messages=msgs, temperature=temperature)
            text = resp.choices[0].message.content
            in_tok = resp.usage.prompt_tokens
            out_tok = resp.usage.completion_tokens
        
        elif provider == "anthropic":
            resp = self._clients["anthropic"].messages.create(
                model=model, max_tokens=2048,
                system=system or "You are a helpful assistant.",
                messages=[{"role": "user", "content": message}],
                temperature=temperature)
            text = resp.content[0].text
            in_tok = resp.usage.input_tokens
            out_tok = resp.usage.output_tokens
        
        latency = (time.time() - start) * 1000
        cost = self._calc_cost(model, in_tok, out_tok)
        
        return LLMResponse(text=text, provider=provider, model=model,
                          input_tokens=in_tok, output_tokens=out_tok,
                          latency_ms=latency, cost_usd=cost)

# ── Use it! Same code, any provider ──
llm = UnifiedLLM()

# Switch providers by changing one string:
for model in ["gemini-2.0-flash", "gpt-4o-mini", "claude-haiku-3-5"]:
    resp = llm.chat("What is RAG in one sentence?", model=model)
    print(f"  [{resp.provider:9s}] {resp.latency_ms:6.0f}ms  ${resp.cost_usd:.5f}  {resp.text[:80]}")


> **What just happened?** We built a UnifiedLLM class with ONE .chat() method that works with any provider. Change the model string from "gemini-2.0-flash" to "gpt-4o" to "claude-sonnet-4-20250514" — same code, same response format, same cost tracking. The LLMResponse dataclass normalizes everything: text, provider name, token counts, latency, and cost. Your app never touches provider-specific code.


### Step 3 · Fallback Routing — Automatic Failover

If Provider A fails, automatically try Provider B. If B fails, try C. Zero downtime.

**`part3_fallback_router.py`** · _Block 5 of 8_

FALLBACK ROUTER: Automatic failover + retry — Try primary → fallback 1 → fallback 2


In [ ]:
# =============================================
# FALLBACK ROUTER: Automatic failover + retry
# Try primary → fallback 1 → fallback 2
# with exponential backoff retry per provider.
# =============================================

import time, logging

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("llm_router")

class FallbackRouter:
    """Route requests with automatic fallback and retry."""
    
    def __init__(self, llm: UnifiedLLM,
                 chain: list[str] = None,
                 max_retries: int = 2,
                 timeout_ms: float = 15000):
        self.llm = llm
        self.chain = chain or ["gemini-2.0-flash", "gpt-4o-mini", "claude-haiku-3-5"]
        self.max_retries = max_retries
        self.timeout_ms = timeout_ms
        self.stats = {model: {"success": 0, "fail": 0} for model in self.chain}
    
    def _try_with_retry(self, message, model, system, temperature):
        """Try a model with exponential backoff retry."""
        for attempt in range(self.max_retries):
            try:
                resp = self.llm.chat(message, model=model,
                                    system=system, temperature=temperature)
                
                # Check latency threshold
                if resp.latency_ms > self.timeout_ms:
                    log.warning(f"{model}: too slow ({resp.latency_ms:.0f}ms)")
                    raise TimeoutError("Latency exceeded threshold")
                
                self.stats[model]["success"] += 1
                return resp
            
            except Exception as e:
                wait = 2 ** attempt  # exponential backoff: 1s, 2s, 4s...
                log.warning(f"{model} attempt {attempt+1} failed: {e}. Retry in {wait}s")
                time.sleep(wait)
        
        self.stats[model]["fail"] += 1
        return None
    
    def route(self, message: str, system: str = "",
             temperature: float = 0.3) -> LLMResponse:
        """Try each model in the chain until one succeeds."""
        
        for model in self.chain:
            log.info(f"Trying: {model}")
            resp = self._try_with_retry(message, model, system, temperature)
            
            if resp:
                if model != self.chain[0]:
                    log.info(f"Used fallback: {model}")
                return resp
        
        raise RuntimeError("All providers failed!")
    
    def health_report(self):
        print("\nProvider Health Report:")
        for model, stats in self.stats.items():
            total = stats["success"] + stats["fail"]
            rate = stats["success"] / total * 100 if total > 0 else 0
            status = "🟢" if rate > 90 else "🟡" if rate > 50 else "🔴"
            print(f"  {status} {model:30s}  {stats['success']}/{total} ({rate:.0f}%)")

# ── Use it ──
router = FallbackRouter(llm)

# This automatically falls back if the primary fails
response = router.route(
    "What are the benefits of vector databases for RAG?",
    system="You are a senior AI engineer. Be concise.",
)

print(f"\nProvider used: {response.provider}")
print(f"Latency: {response.latency_ms:.0f}ms")
print(f"Cost: ${response.cost_usd:.5f}")
print(f"Answer: {response.text[:150]}...")

router.health_report()


> **What just happened?** We built a FallbackRouter with: (1) ordered chain — try Gemini first, then OpenAI, then Claude. (2) retry with exponential backoff — if a request fails, wait 1s and retry; if it fails again, wait 2s and retry. (3) latency threshold — if a provider responds but is too slow (>15 seconds), treat it as failed and try the next one. (4) health tracking — counts successes and failures per provider so you can monitor reliability. Users never see downtime. The failover happens in milliseconds.


### Step 4 · Smart Routing — Best Model for Each Task

Different tasks need different strengths. Route each task to the best provider for it.

**`part4_smart_router.py`** · _Block 6 of 8_

SMART ROUTER: Send each task to the BEST model — for that specific task type.


In [ ]:
# =============================================
# SMART ROUTER: Send each task to the BEST model
# for that specific task type.
# =============================================

class SmartRouter:
    """Route tasks to the best provider based on task type."""
    
    # Which model is best at what?
    ROUTING_TABLE = {
        "code":         {"primary": "gpt-4o",            "fallback": "gemini-2.5-pro"},
        "long_doc":     {"primary": "gemini-2.0-flash",  "fallback": "claude-sonnet-4-20250514"},
        "reasoning":    {"primary": "claude-sonnet-4-20250514", "fallback": "gpt-4o"},
        "simple":       {"primary": "gemini-2.0-flash",  "fallback": "gpt-4o-mini"},
        "creative":     {"primary": "claude-sonnet-4-20250514", "fallback": "gpt-4o"},
        "structured":   {"primary": "gemini-2.0-flash",  "fallback": "gpt-4o-mini"},
    }
    
    def __init__(self, llm: UnifiedLLM):
        self.llm = llm
        self.fallback_router = FallbackRouter(llm)
    
    def classify_task(self, message: str) -> str:
        """Detect what kind of task this is (cheap heuristic)."""
        msg = message.lower()
        
        if any(w in msg for w in ["code", "function", "debug", "python", "javascript", "api", "error"]):
            return "code"
        if any(w in msg for w in ["summarize", "document", "analyze this", "read this"]):
            return "long_doc"
        if any(w in msg for w in ["think", "reason", "pros and cons", "compare", "ethics"]):
            return "reasoning"
        if any(w in msg for w in ["write", "story", "creative", "poem", "essay"]):
            return "creative"
        if any(w in msg for w in ["json", "extract", "classify", "format"]):
            return "structured"
        return "simple"
    
    def route(self, message: str, system: str = "", **kwargs) -> LLMResponse:
        """Automatically route to the best model for this task."""
        task = self.classify_task(message)
        routing = self.ROUTING_TABLE[task]
        
        chain = [routing["primary"], routing["fallback"]]
        self.fallback_router.chain = chain
        
        resp = self.fallback_router.route(message, system=system, **kwargs)
        print(f"  Task: {task:12s} → Model: {resp.model:30s} (${resp.cost_usd:.5f})")
        return resp

# ── Test with different task types ──
smart = SmartRouter(llm)

tasks = [
    "Write a Python function that sorts a list using merge sort.",
    "Summarize this 50-page document about climate change.",
    "Compare the pros and cons of microservices vs monolith architecture.",
    "Write a short story about a robot discovering music.",
    "Extract product name and price from this review as JSON.",
    "What is the capital of France?",
]

print("Smart routing results:\n")
for task in tasks:
    smart.route(task)


> **What just happened?** We built a SmartRouter that classifies each request by task type (code, document, reasoning, creative, structured, simple) and routes it to the best model. Coding questions go to GPT-4o. Long document analysis goes to Gemini (1M token context). Nuanced reasoning goes to Claude. Simple questions go to the cheapest model. Each route has a fallback in case the primary is down. You get the best quality AND lowest cost for every request type.


### Step 5 · Cost Tracking — Know Where Every Dollar Goes

Monitor spend per provider, per task, per hour. Catch cost spikes before they hit your wallet.

**`part5_cost_tracker.py`** · _Block 7 of 8_

COST TRACKER: Real-time spend monitoring


In [ ]:
# =============================================
# COST TRACKER: Real-time spend monitoring
# =============================================

from collections import defaultdict
from datetime import datetime

class CostTracker:
    """Track LLM spending per provider, model, and time period."""
    
    def __init__(self, daily_budget_usd: float = 10.0):
        self.daily_budget = daily_budget_usd
        self.records = []
    
    def log(self, response: LLMResponse):
        self.records.append({
            "timestamp": datetime.now(),
            "provider": response.provider,
            "model": response.model,
            "input_tokens": response.input_tokens,
            "output_tokens": response.output_tokens,
            "cost": response.cost_usd,
            "latency_ms": response.latency_ms,
        })
        
        # Budget alert
        today_cost = self.today_spend()
        if today_cost > self.daily_budget * 0.8:
            print(f"  ⚠️ BUDGET WARNING: ${today_cost:.2f} / ${self.daily_budget:.2f} today")
    
    def today_spend(self):
        today = datetime.now().date()
        return sum(r["cost"] for r in self.records if r["timestamp"].date() == today)
    
    def report(self):
        print("\n💰 Cost Report")
        print("─" * 50)
        
        # Per provider
        by_provider = defaultdict(lambda: {"cost": 0, "calls": 0, "tokens": 0})
        for r in self.records:
            p = by_provider[r["provider"]]
            p["cost"] += r["cost"]
            p["calls"] += 1
            p["tokens"] += r["input_tokens"] + r["output_tokens"]
        
        print(f"\n{'Provider':<12} {'Calls':>6} {'Tokens':>10} {'Cost':>10} {'Avg/call':>10}")
        total_cost = 0
        for prov, data in sorted(by_provider.items()):
            avg = data["cost"] / data["calls"] if data["calls"] else 0
            print(f"  {prov:<12} {data['calls']:>4} {data['tokens']:>8,} ${data['cost']:>8.4f} ${avg:>8.5f}")
            total_cost += data["cost"]
        
        print(f"\n  Total: ${total_cost:.4f}  |  Today: ${self.today_spend():.4f} / ${self.daily_budget:.2f}")

# ── Wire it up with the router ──
tracker = CostTracker(daily_budget_usd=5.0)

for task in tasks:
    resp = smart.route(task)
    tracker.log(resp)

tracker.report()


> **What just happened?** We built a CostTracker that logs every API call with timestamps, tokens, cost, and latency. It provides: per-provider cost breakdown, daily spend tracking, and budget alerts (warning at 80% of daily limit). The report shows calls, total tokens, total cost, and average cost per call per provider. This is how production AI companies keep costs predictable — you know exactly how much each provider costs per call and can optimize routing to minimize spend.


### Step 6 · Project: Production Multi-Provider System

Everything combined: unified wrapper + smart routing + fallback + cost tracking + health checks.

**`project_production_system.py`** · _Block 8 of 8_

PRODUCTION MULTI-PROVIDER SYSTEM — All components wired together.


In [ ]:
# =============================================
# PRODUCTION MULTI-PROVIDER SYSTEM
# All components wired together.
# Drop this into any Python app.
# =============================================

class ProductionLLM:
    """Production-grade LLM client with multi-provider support."""
    
    def __init__(self, daily_budget: float = 10.0):
        self.llm = UnifiedLLM()
        self.router = SmartRouter(self.llm)
        self.tracker = CostTracker(daily_budget_usd=daily_budget)
    
    def ask(self, message: str, system: str = "",
           model: str = None, temperature: float = 0.3) -> LLMResponse:
        """
        The ONE method you use in your app.
        
        - model=None → auto-route to best model for the task
        - model="gpt-4o" → use specific model with fallback
        """
        
        # Check budget BEFORE making the call
        if self.tracker.today_spend() > self.tracker.daily_budget:
            raise RuntimeError("Daily budget exceeded! Call tracker.report() for details.")
        
        # Route the request
        if model:
            # Specific model requested → use with fallback
            resp = self.router.fallback_router.route(message, system=system, temperature=temperature)
        else:
            # Auto-route to best model for this task
            resp = self.router.route(message, system=system, temperature=temperature)
        
        # Track cost
        self.tracker.log(resp)
        return resp
    
    def report(self):
        self.tracker.report()
        self.router.fallback_router.health_report()

# ── Usage in your app ──
ai = ProductionLLM(daily_budget=5.0)

# Auto-route (default — best model picked automatically)
resp = ai.ask("Write a Python function to calculate compound interest.")
print(f"Answer: {resp.text[:100]}...")

# Specific model (with automatic fallback if it fails)
resp = ai.ask("What is the meaning of life?", model="gemini-2.0-flash")
print(f"Answer: {resp.text[:100]}...")

# With system prompt
resp = ai.ask(
    "Explain transformers to a 10-year-old.",
    system="You are a friendly teacher who uses Indian cricket analogies."
)
print(f"Answer: {resp.text[:100]}...")

# Get the full dashboard
ai.report()

print("""
What ProductionLLM gives you:
  ✅ One .ask() method for everything
  ✅ Auto-routing to the best model per task type
  ✅ Automatic fallback if any provider goes down
  ✅ Retry with exponential backoff
  ✅ Latency monitoring (skips slow providers)
  ✅ Real-time cost tracking with budget alerts
  ✅ Health dashboard showing reliability per provider
  
This is what production AI applications use internally.
""")


> **What just happened?** We combined everything into one ProductionLLM class with a single .ask() method. No model specified → auto-routes to the best model for the task. Model specified → uses it with automatic fallback. Budget exceeded → blocks the request before it costs money. Every call is tracked with cost, latency, and provider reliability. The .report() method shows a full dashboard. This is the foundation of every production AI application.


---

## Wrap-up

You've walked through all 8 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
